# 03 — Feature Engineering

In [1]:
import os
from pathlib import Path
import pandas as pd

# --- Project Path Setup ---
# Determine project root reliably whether run from project root or notebooks folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)

PROCESSED_FILE = PROJECT_ROOT / "data" / "processed" / "machine_failure_cleaned.csv"

# 1. Verify that Step 2 cleaned dataset exists before reading
if not os.path.exists("data/processed/machine_failure_cleaned.csv"):
    raise FileNotFoundError(
        "Cleaned dataset 'data/processed/machine_failure_cleaned.csv' not found! "
        "Please run '02_data_cleaning.ipynb' first before running feature engineering."
    )

# Load the cleaned dataset
df = pd.read_csv("data/processed/machine_failure_cleaned.csv")

# 2. Drop non-predictive identifiers and data leakage columns
drop_cols = ["UDI", "Product ID", "TWF", "HDF", "PWF", "OSF", "RNF"]
df = df.drop(columns=drop_cols)
print("Dropped non-predictive and leakage columns:", drop_cols)

# 3. One-hot encode the categorical 'Type' feature
df = pd.get_dummies(df, columns=["Type"])
print("\nColumns after encoding Type:\n", df.columns.tolist())

# 4. Create additional physical domain features
# Feature 1: Temperature difference between process and air
df["Temperature difference [K]"] = df["Process temperature [K]"] - df["Air temperature [K]"]

# Feature 2: Mechanical power indicator (Torque * Rotational speed)
df["Power [W]"] = df["Torque [Nm]"] * df["Rotational speed [rpm]"]

# Feature 3: Cumulative mechanical strain on cutting tool (Tool wear * Torque)
df["Wear-Torque product [minNm]"] = df["Tool wear [min]"] * df["Torque [Nm]"]

# Display summary of the newly engineered features
new_features = ["Temperature difference [K]", "Power [W]", "Wear-Torque product [minNm]"]
print("\nEngineered Features Summary:\n", df[new_features].describe())

# 5. Check correlation between engineered features and Machine failure
print("\nCorrelation of new features with Machine failure:\n",
      df[new_features + ["Machine failure"]].corr()["Machine failure"])

# 6. Display final dataset columns and shape
print("\nFinal feature columns:", df.columns.tolist())
print("Final dataset shape:", df.shape)

# 7. Create directory if necessary and save feature-engineered dataset
os.makedirs("data/processed", exist_ok=True)
df.to_csv("data/processed/machine_failure_features.csv", index=False)

# Verify the file was created
features_created = os.path.exists("data/processed/machine_failure_features.csv")
print("\nFile exists check:", features_created)
if not features_created:
    raise FileNotFoundError("Failed to create 'data/processed/machine_failure_features.csv'.")
print("Saved features dataset to: data/processed/machine_failure_features.csv")

Dropped non-predictive and leakage columns: ['UDI', 'Product ID', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

Columns after encoding Type:
 ['Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]', 'Machine failure', 'Type_H', 'Type_L', 'Type_M']

Engineered Features Summary:
        Temperature difference [K]     Power [W]  Wear-Torque product [minNm]
count                10000.000000  10000.000000                 10000.000000
mean                    10.000630  59967.147040                  4314.664550
std                      1.001094  10193.093881                  2826.567692
min                      7.600000  10966.800000                     0.000000
25%                      9.300000  53105.400000                  1963.650000
50%                      9.800000  59883.900000                  4012.950000
75%                     11.000000  66873.750000                  6279.000000
max                     12.100000  99980.400000              